# Aggregate, Annotate, Normalize, and Feature Select

This notebook will run all these above operations on a single-cell file or well-aggregated file.

We will have the option to run or not 

In [ ]:
path = r"path_to_environment"

In [2]:
import pandas as pd
import numpy as np
import pycytominer
import easygui as eg
import os
import sys
import gc
sys.path.append(path)
from scripts_notebooks_fossa.profiles import generate_profiles
%load_ext autoreload
%autoreload 2

# 0) Inputs

In [7]:
def stringToBool(s):
    return s.lower() == 'yes'

profile = eg.fileopenbox(msg="Choose a file with samples and their features", default=r"D:")
print('Filename', profile)

project_name = input('Provide the name of this project:')
print('Project name:', project_name)

metadata_question = input("If you need to annotate your dataset with an external file, write yes and press enter. If already annotated, answer no and press enter.")
metadata_answer = stringToBool(metadata_question)

if metadata_answer:
    platemap = eg.fileopenbox(msg="Choose a map (csv or txt file) with plates names and metadata filenames", default=r"G:")
    platemap_path = os.path.split(platemap)[0]
    print('Platemap file selected', platemap)
    
    # Read platemap using pandas (assuming it's either CSV or TXT)
    barcode_df = pd.read_csv(platemap)
else:
    barcode_df = None  # You can assign a default value or leave it as None, depending on your needs

cells_that_run = []
cells_that_run.append(project_name)

Filename G:\Meu Drive\Doutorado Thais\Github_Analysis\Dormant_NonDormant\SampleRetrival\Dormant_x_Nondormant_SingleCells.csv
Project name: ao
Platemap file selected G:\Meu Drive\Doutorado Thais\pHD_analysisfolder\workspace\metadata\platemaps\Dormant_x_normal\metadata\barcode_platemap.csv


In [8]:
barcode_df

,Assay_Plate_Barcode,Plate_Map_Name
0,240319_144243_Plate_1_,layout_day1.txt
1,240321_153415_Plate_1_,layout_day2.txt
2,240404_134523_Plate_1_,layout_day3.txt


In [9]:
gc.collect()


971

## 1) Import extracted features file (single cell or well-aggregated)

In [10]:
df = pd.read_csv(profile)
df.head()

,Metadata_Plate,Metadata_QCFlag,Metadata_QCFlag.1,Metadata_Site,Metadata_Well,Nuclei_ImageNumber,Nuclei_ObjectNumber,Nuclei_AreaShape_Area,Nuclei_AreaShape_BoundingBoxArea,Nuclei_AreaShape_BoundingBoxMaximum_X,...,Cells_Texture_Variance_AOPI_10_02_256,Cells_Texture_Variance_AOPI_10_03_256,Cells_Texture_Variance_AOPI_20_00_256,Cells_Texture_Variance_AOPI_20_01_256,Cells_Texture_Variance_AOPI_20_02_256,Cells_Texture_Variance_AOPI_20_03_256,Cells_Texture_Variance_AOPI_5_00_256,Cells_Texture_Variance_AOPI_5_01_256,Cells_Texture_Variance_AOPI_5_02_256,Cells_Texture_Variance_AOPI_5_03_256
0,240321_153415_Plate_1_,0,0,11,D10,3,1,2609,3534,150,...,41.960698,42.253443,42.380039,44.859571,43.866346,45.269071,40.532123,40.923713,41.380130,41.272143
1,240321_153415_Plate_1_,0,0,11,D10,3,2,1837,2255,513,...,214.205638,216.504315,207.841614,207.263831,222.108959,244.118014,206.658726,209.753136,207.986281,208.201253
2,240321_153415_Plate_1_,0,0,11,D10,3,3,2466,3190,82,...,53.725273,51.841565,64.554173,59.322037,51.237739,55.302862,63.215495,60.704996,60.425950,59.798456
3,240321_153415_Plate_1_,0,0,11,D10,3,4,2002,2700,498,...,212.610204,204.039853,188.735030,163.388927,209.904213,226.798017,212.344328,215.252342,219.046176,212.632572
4,240321_153415_Plate_1_,0,0,11,D10,3,5,1835,2322,558,...,340.827507,338.439665,358.311890,364.678780,372.306522,388.147483,322.325372,339.392508,327.577464,323.398845


## 2) Generate profile

### 2A) Aggregate

- Run the next cell to list the metadata columns, and copy them to choose from which metadata information you'd like to join the rows on (strata)

- If using **already aggregated data by plates and wells**, skip to 2B.

In [11]:
pycytominer.cyto_utils.infer_cp_features(df, metadata=True)

['Metadata_Plate',
 'Metadata_QCFlag',
 'Metadata_QCFlag.1',
 'Metadata_Site',
 'Metadata_Well',
 'Metadata_Plate.1',
 'Metadata_QCFlag.2',
 'Metadata_QCFlag.1.1',
 'Metadata_Site.1',
 'Metadata_Well.1',
 'Metadata_Plate.2',
 'Metadata_QCFlag.3',
 'Metadata_QCFlag.1.2',
 'Metadata_Site.2',
 'Metadata_Well.2']

In [12]:
df1 = df

In [13]:
del df

In [14]:
gc.collect()

0

- As a default we are using **Metadata_Plate and Metadata_Well** to join the rows, using the **'median'** operation. 

### 2B) Annotate 

In [15]:
df = df1

In [16]:
del df1

In [17]:
gc.collect()

0

- Run the following to generate a plate list based on Metadata_Plate column.

In [18]:
plate_list = df['Metadata_Plate'].unique().tolist()

In [19]:
print(plate_list)

['240321_153415_Plate_1_', '240404_134523_Plate_1_']


- All metadata must be organized such:
    ```
    |- metadata 
    |   |   |- <barcode_platemap.csv> 
    |   |- platemap
    |   |   |- <platemap_1.csv>
    |   |   |- <platemap_2.csv>
    ```

- Run the next cell to annotate the profiles. 

In [20]:
df_temp_list = []
for pl in plate_list:
    df_plate = df.loc[df['Metadata_Plate'] == pl]
    #deal with metadata information
    barcode_map = barcode_df[barcode_df['Assay_Plate_Barcode'] == pl]
    index_map = barcode_map.index[0]
    metadata_filename = barcode_df['Plate_Map_Name'][index_map]
    metadata = pd.read_csv(platemap_path + r'/platemap/' + metadata_filename)
    #annotate
    df_temp = pycytominer.annotate(profiles = df_plate, platemap = metadata, join_on =  ["Metadata_well_position", "Metadata_Well"])
    df_temp_list.append(df_temp)
    print('Shape of each plate ', df_temp.shape)
df = pd.concat(df_temp_list, axis=0)

Shape of each plate  (1129, 1835)
Shape of each plate  (10730, 1835)


In [21]:
df

,Metadata_plate_map_name,Metadata_cell,Metadata_compound,Metadata_concentration_uM,Metadata_control_type,Metadata_Plate,Metadata_QCFlag,Metadata_QCFlag.1,Metadata_Site,Metadata_Well,...,Cells_Texture_Variance_AOPI_10_02_256,Cells_Texture_Variance_AOPI_10_03_256,Cells_Texture_Variance_AOPI_20_00_256,Cells_Texture_Variance_AOPI_20_01_256,Cells_Texture_Variance_AOPI_20_02_256,Cells_Texture_Variance_AOPI_20_03_256,Cells_Texture_Variance_AOPI_5_00_256,Cells_Texture_Variance_AOPI_5_01_256,Cells_Texture_Variance_AOPI_5_02_256,Cells_Texture_Variance_AOPI_5_03_256
0,layout_day2,MCF7,CoCl2,300,trt,240321_153415_Plate_1_,0,0,1,D2,...,82.315439,83.777692,87.644268,82.921315,86.507336,79.152376,81.124492,81.709004,80.984935,81.306505
1,layout_day2,MCF7,CoCl2,300,trt,240321_153415_Plate_1_,0,0,1,D2,...,47.876735,48.722313,48.930022,47.019687,49.443581,57.347669,46.695459,47.275254,47.679716,47.509463
2,layout_day2,MCF7,CoCl2,300,trt,240321_153415_Plate_1_,0,0,1,D2,...,150.158007,159.580331,170.444522,172.213394,160.969366,178.721217,152.579596,152.274162,150.308046,153.482872
3,layout_day2,MCF7,CoCl2,300,trt,240321_153415_Plate_1_,0,0,1,D2,...,65.545259,68.035121,66.934904,73.745894,74.303297,76.023144,64.095451,64.546347,63.412176,65.453489
4,layout_day2,MCF7,CoCl2,300,trt,240321_153415_Plate_1_,0,0,1,D2,...,38.051508,37.853836,41.003809,41.004686,39.378305,40.059765,38.593240,38.460073,37.954065,38.078815
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10725,layout_day3,MCF7,Non-treated,0,negcon,240404_134523_Plate_1_,0,0,9,G11,...,28.984989,30.196354,30.341514,34.436630,33.368958,33.722833,27.652810,27.666757,27.724166,28.824822
10726,layout_day3,MCF7,Non-treated,0,negcon,240404_134523_Plate_1_,0,0,9,G11,...,698.398166,693.848008,746.781101,796.290040,745.628961,738.160756,655.270973,683.052301,659.160632,658.485412
10727,layout_day3,MCF7,Non-treated,0,negcon,240404_134523_Plate_1_,0,0,9,G11,...,363.202645,373.956566,378.808516,275.814655,365.158304,316.685097,341.973661,349.578434,346.181169,351.722953
10728,layout_day3,MCF7,Non-treated,0,negcon,240404_134523_Plate_1_,0,0,9,G11,...,2567.905469,2621.492931,2907.170237,2700.775474,2932.595072,3008.993413,2401.193515,2469.846233,2395.196859,2419.281795


Split cell count in a separate df

### 2C) Normalize

- Normalize the dataset, per-plate basis, to **samples = all** or **samples = negcon**.
- CHOOSE one or the other below.

#### Normalize TO NEGCON => run next cell

In [22]:
df_temp_list = []
for pl in plate_list:
    df_temp = df.loc[df['Metadata_Plate'] == pl]
    print(df_temp.shape)
    df_norm_temp = pycytominer.normalize(df_temp, method = 'mad_robustize', mad_robustize_epsilon = 0, samples = "Metadata_control_type	 == 'negcon'") 
    df_temp_list.append(df_norm_temp)
df_norm2 = pd.concat(df_temp_list, axis=0)
cells_that_run.append('normalized_negcon')

(1129, 1835)
(10730, 1835)


c:\Users\Lab\miniconda3\envs\bioimage_scripts\lib\site-packages\scipy\stats\_stats_py.py:2894: RuntimeWarning: invalid value encountered in subtract
  mad = np.median(np.abs(x - med))
c:\Users\Lab\miniconda3\envs\bioimage_scripts\lib\site-packages\scipy\stats\_stats_py.py:2894: RuntimeWarning: invalid value encountered in subtract
  mad = np.median(np.abs(x - med))


In [23]:
#porque o codigo abaixo estava dando erro de versao
np.bool = bool

In [24]:
df_selected = pycytominer.feature_select(df_norm2, operation = ['correlation_threshold', 'variance_threshold', 'drop_na_columns','drop_outliers'], outlier_cutoff = 500) 
print('Number of columns removed:', df_norm2.shape[1] - df_selected.shape[1])
print('Percentage of columns removed:',100 - ((df_selected.shape[1]*100)/df_norm2.shape[1]))
cells_that_run.append('feature_select')

Number of columns removed: 1243
Percentage of columns removed: 67.73841961852861


# Export

In [25]:
output_path = eg.diropenbox(msg="Choose an output folder", default=r"D:")
print('Path to save the profile', output_path)

Path to save the profile G:\Meu Drive\Doutorado Thais\Github_Analysis\Dormant_NonDormant\Normalized


In [26]:
output_name = '_'.join(cells_that_run)
df_selected.to_csv(output_path + r'/' + 'Dormant_vs_Nondorm_SingleCell_FEATSELECTED' + '.csv')
print('Successfully exported to:', output_path + r'/' + output_name + '.csv')

Successfully exported to: G:\Meu Drive\Doutorado Thais\Github_Analysis\Dormant_NonDormant\Normalized/ao_normalized_negcon_feature_select.csv


In [22]:
output_name = '_'.join(cells_that_run)
df_norm2.to_csv(output_path + r'/' + 'Dormant_vs_Nondorm_SingleCell_NORMALIZED' + '.csv')
print('Successfully exported to:', output_path + r'/' + output_name + '.csv')

Successfully exported to: G:\Meu Drive\Doutorado Thais\Github_Analysis\Dormant_NonDormant\Normalized/ao_normalized_negcon.csv
